# Route definition 

In [2]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Configuración de rutas (asegúrate de que ROOT suba a la raíz)
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from config.paths import RAW_DIR, PROCESSED_DIR

YEARS = [2018, 2024]


# Imputation of IVA

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuración de prefijos IVA y códigos de informalidad por año
YEAR_CONFIG = {
    2018: {
        # Prefijos: C (Limpieza), D (Cuidados), F (Comunicaciones/Combustible), 
        # H (Vestido), I (Muebles), K (Enseres), L (Esparcimiento), M (Transporte), N (Otros)
        "iva_prefixes": ('C', 'D', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N'),
        "informal_locs": [1, 2, 3, 17], # Códigos 2018
        "clave_len": 4
    },
    2024: {
        # Prefijos COICOP: 02 (Alcohol), 03 (Vestido), 05 (Muebles), 07 (Transporte), 
        # 08 (Comunicación), 09 (Recreación), 11 (Restaurantes), 12 (Otros)
        "iva_prefixes": ('012', '02', '03', '05', '07', '08', '09', '11', '12'),
        "informal_locs": [1, 2, 3, 17], # Generalmente se mantienen, verificar en FD
        "clave_len": 6
    }
}

def fix_and_calculate_iva(year, raw_dir):
    print(f"🛠️  Procesando ENIGH {year}...")
    
    config = YEAR_CONFIG[year]
    
    # 1. Carga de datos
    df_gastos = pd.read_csv(raw_dir / f"gastoshogar_{year}.csv", 
                            dtype={'folioviv': str, 'foliohog': str, 'clave': str}, 
                            low_memory=False)
    df_conc = pd.read_csv(raw_dir / f"concentradohogar_{year}.csv", 
                          dtype={'folioviv': str, 'foliohog': str}, 
                          low_memory=False)

    # 2. Procesamiento de Gastos
    df_gastos['gasto_tri'] = pd.to_numeric(df_gastos['gasto_tri'], errors='coerce').fillna(0)
    
    # Filtrar solo gastos monetarios (G1)
    df_gastos = df_gastos[df_gastos['tipo_gasto'] == 'G1'].copy()
    
    # Identificar si lleva IVA basado en los prefijos del año correspondiente
    df_gastos['lleva_iva'] = df_gastos['clave'].str.startswith(config["iva_prefixes"]).astype(int)
    
    # Identificar formalidad 
    df_gastos['es_formal'] = (~df_gastos['lugar_comp'].isin(config["informal_locs"])).astype(int)
    
    # Cálculo del IVA por partida: Gasto * (0.16 / 1.16)
    # $Factor_{IVA} = \frac{0.16}{1.16} \approx 0.137931$ 
    factor_iva = 0.137931034
    df_gastos['iva_partida'] = np.where((df_gastos['lleva_iva'] == 1) & (df_gastos['es_formal'] == 1),
                                        df_gastos['gasto_tri'] * factor_iva, 0)
    
    # Agrupar IVA por hogar
    iva_hogar = df_gastos.groupby(['folioviv', 'foliohog'])['iva_partida'].sum().reset_index()
    iva_hogar.rename(columns={'iva_partida': 'IVA_HH'}, inplace=True)

    # 3. Preparar Concentrado
    cols_num = ['gasto_mon', 'alimentos', 'limpieza', 'cuidados', 'ing_cor', 'factor']
    for col in cols_num:
        df_conc[col] = pd.to_numeric(df_conc[col], errors='coerce').fillna(0)
    
    # Base de gasto estructural (excluyendo lo básico que suele ser tasa 0% o exento)
    df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (df_conc['alimentos'] + df_conc['limpieza'] + df_conc['cuidados'])

    # 4. Merge y cálculo de tasa efectiva
    df_final = pd.merge(df_conc, iva_hogar, on=['folioviv', 'foliohog'], how='left').fillna(0)
    
    # Cálculo del IVA atribuible al ingreso (proporcional)
    # Evitamos división por cero con np.where
    df_final['IVA_HH_A'] = np.where(df_final['Gasto_HH_tri'] > 0,
                                    (df_final['IVA_HH'] / df_final['Gasto_HH_tri']) * df_final['ing_cor'],
                                    0)
    
    return df_final.copy()

# Ejecución del Loop
results = {}
for year in [2018, 2024]:
    df_result = fix_and_calculate_iva(year, RAW_DIR)
    
    # Cálculo de Deciles
    df_result = df_result.sort_values("ing_cor")
    df_result["cumsum_factor"] = df_result["factor"].cumsum()
    df_result["decil"] = pd.qcut(df_result["cumsum_factor"], 10, labels=range(1, 11))
    
    # Resumen ponderado
    summary = df_result.groupby("decil").agg(
        ingreso_medio=("ing_cor", lambda x: np.average(x, weights=df_result.loc[x.index, "factor"])),
        iva_medio=("IVA_HH_A", lambda x: np.average(x, weights=df_result.loc[x.index, "factor"])),
        hogares_representados=("factor", "sum")
    ).reset_index()
    
    summary["carga_iva_pct"] = (summary["iva_medio"] / summary["ingreso_medio"]) * 100
    results[year] = summary

    print(f"\n{'='*20} ENIGH {year} {'='*20}")
    display(summary.style.format({
        "ingreso_medio": "${:,.2f}", "iva_medio": "${:,.2f}", 
        "carga_iva_pct": "{:.2f}%", "hogares_representados": "{:,.0f}"
    }))

🛠️  Procesando ENIGH 2018...

==================== ENIGH 2018 ====================


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_2943/3005975830.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (df_conc['alimentos'] + df_conc['limpieza'] + df_conc['cuidados'])
/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_2943/3005975830.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['IVA_HH_A'] = np.where(df_final['Gasto_HH_tri'] > 0,


,decil,ingreso_medio,iva_medio,hogares_representados,carga_iva_pct
0,1,"$8,716.22","$1,658.35","3,077,839",19.03%
1,2,"$15,291.60","$2,251.26","3,127,662",14.72%
2,3,"$20,324.76","$2,223.45","3,221,401",10.94%
3,4,"$25,314.82","$2,806.68","3,299,920",11.09%
4,5,"$30,614.86","$3,098.35","3,412,828",10.12%
5,6,"$36,961.89","$3,629.89","3,460,309",9.82%
6,7,"$44,727.99","$4,388.86","3,501,973",9.81%
7,8,"$55,585.77","$5,446.05","3,575,937",9.80%
8,9,"$73,631.94","$8,816.83","3,682,880",11.97%
9,10,"$156,510.85","$18,790.49","4,039,766",12.01%


🛠️  Procesando ENIGH 2024...

==================== ENIGH 2024 ====================


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_2943/3005975830.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (df_conc['alimentos'] + df_conc['limpieza'] + df_conc['cuidados'])
/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_2943/3005975830.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['IVA_HH_A'] = np.where(df_final['Gasto_HH_tri'] > 0,


,decil,ingreso_medio,iva_medio,hogares_representados,carga_iva_pct
0,1,"$15,688.53","$5,014.86","3,283,413",31.97%
1,2,"$26,482.16","$3,675.52","3,532,617",13.88%
2,3,"$34,539.90","$4,416.71","3,634,153",12.79%
3,4,"$42,452.03","$5,475.26","3,746,813",12.90%
4,5,"$51,059.62","$6,283.88","3,861,828",12.31%
5,6,"$60,895.15","$7,609.87","3,949,553",12.50%
6,7,"$72,846.22","$9,077.25","3,922,847",12.46%
7,8,"$89,292.18","$12,463.10","4,055,830",13.96%
8,9,"$115,561.16","$14,573.56","4,229,375",12.61%
9,10,"$220,959.36","$34,181.70","4,613,801",15.47%


In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# CONFIGURACIÓN
# ============================================================
# Configuración de rutas (asegúrate de que ROOT suba a la raíz)

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from config.paths import RAW_DIR, PROCESSED_DIR

FACTOR_IVA = 0.16 / 1.16   # 0.137931...

# ============================================================
# 1. Función para clasificar si una clave lleva IVA (gravado 16%)
# ============================================================
def es_gravado_2018(clave):
    """
    Retorna True si el gasto está gravado al 16% y por tanto incluye IVA.
    Base: tabla de gastospersona 2018 (A001-R013, etc.)
    """
    # --- Alimentos ---
    # Toda la despensa es tasa 0%, excepto alimentos preparados fuera del hogar
    if clave.startswith('A'):
        # Alimentos preparados (incluyen IVA)
        if clave in ['A198','A199','A200','A201','A202','A243','A244','A245','A246','A247']:
            return True
        # Agua natural embotellada (A215) y mineral sin sabor (A216): tasa 0%
        # Refrescos, jugos, concentrados y polvos: gravados (A218,A219,A220,A221,A222)
        if clave in ['A218','A219','A220','A221','A222']:
            return True
        # Bebidas alcohólicas y tabaco (A223-A241): gravados
        if clave >= 'A223' and clave <= 'A241':
            return True
        # El resto de alimentos y agua simple: tasa 0% -> no IVA
        return False

    # --- Transporte público terrestre (tasa 0%) ---
    if clave.startswith('B'):
        return False   # Metro, autobús, taxi, etc.

    # --- Limpieza y menaje del hogar (gravado) ---
    if clave.startswith('C'):
        return True

    # --- Cuidado personal y cosméticos (gravado) ---
    if clave.startswith('D'):
        return True

    # --- Educación (servicios exentos, material mixto) ---
    if clave.startswith('E'):
        # Servicios educativos (E001-E013, E018, E020): exentos -> no IVA
        # Material educativo: libros (E022-E024) tasa 0%, papelería gravada
        gravado_papeleria = ['E017','E019','E021','E025','E026']  # equipo, artículos, etc.
        if clave in gravado_papeleria:
            return True
        # E027-E034 entretenimiento: gravado
        if clave >= 'E027' and clave <= 'E034':
            return True
        return False

    # --- Comunicaciones y combustibles (gravado) ---
    if clave.startswith('F'):
        return True

    # --- Vivienda (alquiler exento, servicios mixtos) ---
    if clave.startswith('G'):
        # Alquiler de vivienda y cuotas (G001-G008, G101-G106): exentos
        # Energía, agua, gas (G009-G016): gas y electricidad gravados, leña y carbón tasa 0%
        if clave in ['G009','G010','G011','G014']:   # gas LP, petróleo, diesel, combustible p/calentar
            return True
        # Agua (R002) está en prefijo R, no G
        return False

    # --- Vestido y calzado (gravado) ---
    if clave.startswith('H'):
        return True   # incluye uniformes y accesorios

    # --- Muebles, blancos, utensilios, electrodomésticos (gravado) ---
    if clave.startswith('I') or clave.startswith('K'):
        return True

    # --- Salud: medicamentos tasa 0%, servicios exentos, aparatos gravados ---
    if clave.startswith('J'):
        # Aparatos ortopédicos, lentes, etc. (gravados)
        gravado_salud = ['J065','J066','J067']   # anteojos, aparatos sordera, ortopédicos
        if clave in gravado_salud:
            return True
        return False

    # --- Esparcimiento, electrónica, juguetes (gravado) ---
    if clave.startswith('L'):
        return True

    # --- Transporte: vehículos, refacciones, aéreo (gravado); foráneo terrestre tasa 0% ---
    if clave.startswith('M'):
        # Transporte foráneo terrestre (M001, M002): tasa 0%
        if clave in ['M001','M002']:
            return False
        return True

    # --- Otros: servicios profesionales, funerarios, seguros, trámites (gravado) ---
    if clave.startswith('N'):
        return True   # en general gravados; algunos exentos (ej. N007 contribuciones) son raros

    # --- Operaciones financieras, ahorro, préstamos (exentos) ---
    if clave.startswith('Q'):
        return False

    # --- Servicios del hogar: electricidad, agua, gas (algunos ya captados) ---
    if clave.startswith('R'):
        # R001 electricidad: gravado
        # R002 agua: exento
        # R003 gas natural: gravado
        # R004 predial: exento
        # R005-R012 telefonía, internet, TV, tenencia: gravados
        if clave in ['R002','R004']:
            return False
        return True

    # Por defecto, si aparece algo no catalogado, asumimos gravado (seguridad)
    return True


def es_gravado_2024(clave):
    """
    Retorna True si el gasto está gravado al 16%.
    Claves 2024 de 6 dígitos.
    """
    # --- Alimentos preparados fuera del hogar (11xxxx) -> gravado ---
    if clave.startswith('11') and clave[2] == '1':   # 1111xx, 1112xx
        return True
    # --- Resto de alimentos y bebidas no alcohólicas ---
    if clave.startswith('01'):
        # Agua natural embotellada (012501): tasa 0%
        # Refrescos, jugos con azúcar, concentrados, bebidas energéticas: gravados
        gravado_bebidas = ['012601','012602','012604','012903','012904','012905']
        if clave in gravado_bebidas:
            return True
        if clave == '012100':   # jugos de frutas o verduras envasados (con azúcar añadida) -> gravado
            return True
        # La mayoría de alimentos (011xxx) son tasa 0%, salvo excepciones
        # Los alimentos en paquete (1811xx) son tasa 0%
        return False

    # --- Bebidas alcohólicas y tabaco (02xxxx) -> gravado ---
    if clave.startswith('02'):
        return True

    # --- Vestido y calzado (03xxxx) -> gravado ---
    if clave.startswith('03'):
        return True

    # --- Vivienda: alquiler exento (041104,041211,041221), servicios mixtos ---
    if clave.startswith('04'):
        if clave in ['041104','041211','041221']:
            return False
        if clave.startswith('044110') or clave.startswith('045100'):   # agua, electricidad
            # Agua (044110) exento, electricidad (045100) gravado
            return clave.startswith('045')
        if clave.startswith('045221') or clave.startswith('045222') or clave.startswith('045420'):
            return True   # gas LP, leña y carbón tasa 0? En 2024 leña es tasa 0% (combustible sólido). Ajustamos.
        return False

    # --- Muebles, electrodomésticos, menaje (05xxxx) -> gravado ---
    if clave.startswith('05'):
        return True

    # --- Salud (06xxxx): medicamentos tasa 0%, servicios exentos, productos sí ---
    if clave.startswith('06'):
        # Medicamentos y productos farmacéuticos: tasa 0% -> no IVA
        if clave.startswith('0611'):   # 0611xx
            return False
        # Servicios médicos, hospitalarios: exentos (062xxx, 063xxx, 064xxx)
        if clave.startswith('062') or clave.startswith('063') or clave.startswith('064'):
            return False
        return False

    # --- Transporte (07xxxx): público terrestre tasa 0%, aéreo gravado, vehículos/combustibles gravados ---
    if clave.startswith('07'):
        # Transporte público: metro, autobús, taxi, app, tren ligero, cablebús -> 073xxx tasa 0%
        if clave.startswith('073'):
            return False
        # Vehículos, combustibles, mantenimiento: gravados
        return True

    # --- Comunicaciones (08xxxx) -> gravado ---
    if clave.startswith('08'):
        return True

    # --- Recreación y cultura (09xxxx): la mayoría gravado, libros tasa 0% ---
    if clave.startswith('09'):
        # Libros de texto (097111), periódicos (097210), revistas (097220) -> tasa 0%
        if clave in ['097111','097210','097220']:
            return False
        return True

    # --- Educación (10xxxx): servicios exentos, material mixto ---
    if clave.startswith('10'):
        # Colegiaturas e inscripciones (101xxx-105xxx): exentos
        return False

    # --- Restaurantes y alimentos preparados (11xxxx) ya cubierto al inicio ---
    if clave.startswith('11'):
        # 1111xx (desayunos, comidas, cenas, entrecomidas, etc.) -> gravado
        if clave.startswith('1111'):
            return True
        return False   # otros como 112xxx alojamiento (exento en algunos casos, pero se grava)

    # --- Seguros (12xxxx) -> gravado ---
    if clave.startswith('12'):
        return True

    # --- Cuidado personal (13xxxx): gravado ---
    if clave.startswith('13'):
        return True

    # --- Otras erogaciones (17xxxx, 18xxxx): se gravan en su mayoría ---
    if clave.startswith('17') or clave.startswith('18'):
        # 174xxx: educación inicial (exento), 173xxx varios
        return True

    # Por defecto, gravado
    return True


# ============================================================
# 2. Script principal
# ============================================================
output_frames = []

for year in YEARS:
    print(f"Procesando ENIGH {year} (gastospersona)...")
    file_path = RAW_DIR / f"gastospersona_{year}.csv"
    df = pd.read_csv(file_path, dtype={'folioviv':str, 'foliohog':str, 'numren':str, 'clave':str})

    # --- Filtrar solo gastos monetarios ---
    df = df[df['tipo_gasto'] == 'G1'].copy()

    # --- Asegurar que gasto_tri sea numérico ---
    df['gasto_tri'] = pd.to_numeric(df['gasto_tri'], errors='coerce').fillna(0)

    # --- Determinar si la clave está gravada según el año ---
    if year == 2018:
        df['es_gravado'] = df['clave'].apply(es_gravado_2018)
    else:
        df['es_gravado'] = df['clave'].apply(es_gravado_2024)

    # --- Filtro de informalidad (solo si existe la columna 'lugar_comp') ---
    if 'lugar_comp' in df.columns:
        informal_locs = [1,2,3,17]   # mismos códigos que usabas
        df['es_formal'] = (~df['lugar_comp'].isin(informal_locs)).astype(bool)
    else:
        # Si no existe, asumimos que todos los gastos son formales
        df['es_formal'] = True

    # --- Cálculo del IVA incluido en el precio ---
    df['iva'] = np.where(df['es_gravado'] & df['es_formal'],
                         df['gasto_tri'] * FACTOR_IVA, 0.0)

    # --- Agrupar por persona ---
    persona = df.groupby(['folioviv','foliohog','numren'], as_index=False).agg(
        gasto_total = ('gasto_tri', 'sum'),
        gasto_iva   = ('iva', 'sum')
    )
    persona['year'] = year
    output_frames.append(persona)

# --- Unir todos los años y guardar ---
final = pd.concat(output_frames, ignore_index=True)
final = final[['folioviv','foliohog','numren','year','gasto_total','gasto_iva']]


  # Guardar temporal
out_file = PROCESSED_DIR / 'iva_persona_2018_2024.csv'
final.to_csv(out_file, index=False)
print(f"Archivo guardado en: {out_file}")

Procesando ENIGH 2018 (gastospersona)...


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_2943/2837709820.py:238: DtypeWarning: Columns (0: frec_rem, 1: gasto, 2: costo, 3: gasto_tri, 4: gasto_nm, 5: gas_nm_tri) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, dtype={'folioviv':str, 'foliohog':str, 'numren':str, 'clave':str})


Procesando ENIGH 2024 (gastospersona)...
Archivo guardado en: /Users/jesussanchez/Documents/Pre-trabajo/3.-Mapping-the-Fiscal-Burden/fiscal_incidence_project/data/processed/iva_persona_2018_2024.csv
